<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/googlecolab-init/Medical_Assistant_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Address Colab - Github Compatibility for nbformat

In [6]:
import json

with open('Medical_Assistant_Deployment.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    for widget_key in nb['metadata']['widgets']:
        if 'state' not in nb['metadata']['widgets'][widget_key]:
            nb['metadata']['widgets'][widget_key]['state'] = {}

with open('Medical_Assistant_Deployment.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

Objective is Serialize the model, and expose it as an API. Create a basic User Interface, Dockerize and Deploy the entire stack to Huggingface Space.

## Data Description

The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.
- **Store_Type** - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **Installing and Importing the necessary libraries**

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 pipreqs -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)


# Libraries different ensemble classifiers
from sklearn.ensemble import (
    RandomForestRegressor
)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

# Libraries to get different metric scores
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

# To create the pipeline
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline,Pipeline

# To tune different models and standardize
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# **Loading the dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# loading data into a pandas dataframe
retail_data = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Model Deployment/SuperKart/SuperKart.csv")

In [ ]:
# Create a copy of the dataframe
dataset = retail_data.copy()

# **Deployment - Backend**

## Flask API

The API has two endpoints on for **health check** and one for **posting data** from Streamlit user interface for creating predictions


In [ ]:
%%writefile deployment_files/api.py

import pandas as pd
import joblib
from flask import Flask, request, jsonify

app = Flask(__name__)

model = joblib.load("retail_chain_forecast_model_v1_0.joblib")

REQUIRED_FIELDS = [
    "Product_Id", "Product_Weight", "Product_Sugar_Content",
    "Product_Allocated_Area", "Product_Type", "Product_MRP",
    "Store_Id", "Store_Establishment_Year", "Store_Size",
    "Store_Location_City_Type", "Store_Type"
]

def validate_input(data):
    missing = [f for f in REQUIRED_FIELDS if f not in data]
    if missing:
        return False, f"Missing fields: {missing}"
    if not isinstance(data["Product_Weight"], (int, float)):
        return False, "Product_Weight must be numeric"
    if not isinstance(data["Product_MRP"], (int, float)):
        return False, "Product_MRP must be numeric"
    if not isinstance(data["Product_Allocated_Area"], (int, float)):
        return False, "Product_Allocated_Area must be numeric"
    if not isinstance(data["Store_Establishment_Year"], int):
        return False, "Store_Establishment_Year must be an integer"
    return True, None

def build_dataframe(obs):
    return pd.DataFrame([{
        "Product_Id":               obs["Product_Id"],
        "Product_Weight":           float(obs["Product_Weight"]),
        "Product_Sugar_Content":    obs["Product_Sugar_Content"],
        "Product_Allocated_Area":   float(obs["Product_Allocated_Area"]),
        "Product_Type":             obs["Product_Type"],
        "Product_MRP":              float(obs["Product_MRP"]),
        "Store_Id":                 obs["Store_Id"],
        "Store_Establishment_Year": int(obs["Store_Establishment_Year"]),
        "Store_Size":               obs["Store_Size"],
        "Store_Location_City_Type": obs["Store_Location_City_Type"],
        "Store_Type":               obs["Store_Type"],
    }])

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model_loaded": model is not None}), 200

@app.route("/predict", methods=["POST"])
def predict():
    body = request.get_json(force=True, silent=True)
    if body is None:
        return jsonify({"error": "Request body must be valid JSON"}), 400

    observations = body if isinstance(body, list) else [body]
    predictions  = []

    for i, obs in enumerate(observations):
        valid, err = validate_input(obs)
        if not valid:
            return jsonify({"error": f"Observation {i}: {err}"}), 422

        forecast = model.predict(build_dataframe(obs))[0]
        predictions.append({
            "Product_Id":       obs["Product_Id"],
            "Store_Id":         obs["Store_Id"],
            "forecasted_sales": round(float(forecast), 2)
        })

    return jsonify(predictions if isinstance(body, list) else predictions[0]), 200

if __name__ == "__main__":
    # debug=False is critical for production — debug=True enables the reloader
    # which forks the process and breaks Hugging Face's single-process container
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting deployment_files/api.py


## Setting up a Hugging Face Docker Space for the Backend

I created the following public space https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast to deploy user interface and Flask API

# **Deployment - Frontend**

## Streamlit for Interactive UI

In [ ]:
%%writefile deployment_files/app.py

import requests
import streamlit as st
import pandas as pd

# API endpoint to invoke model
API_URL = "http://localhost:5000"

# Streamlit UI configuration
st.set_page_config(page_title="SuperKart Demand Forecast", layout="centered")
st.title("SuperKart — Quarterly Sales Forecast")
st.write(
    "Internal forecasting tool for SuperKart Retail Chain. "
    "Enter store and product details to predict sales revenue for the upcoming quarter."
)

product_types = ['Baking Goods', 'Breads', 'Breakfast', 'Canned', 'Dairy',
       'Frozen Foods', 'Fruits and Vegetables', 'Hard Drinks',
       'Health and Hygiene', 'Household', 'Meat', 'Others', 'Seafood',
       'Snack Foods', 'Soft Drinks', 'Starchy Foods']

product_sugar_content_types = ['Low Sugar', 'No Sugar', 'Regular']

store_loc_types = ['Tier 1', 'Tier 2', 'Tier 3']

store_sizes = ['High', 'Medium', 'Small']

store_types = ['Departmental Store', 'Food Mart', 'Supermarket Type1',
       'Supermarket Type2']


# UI for capturing Store Details
st.subheader("Store Details")
col1, col2 = st.columns(2)
with col1:
    Store_Id   = st.text_input("Store ID", value="OUT004")
    Store_Size = st.selectbox("Store Size", options=store_sizes)
    Store_Type = st.selectbox("Store Type", options=store_types)
with col2:
    Store_Establishment_Year = st.number_input(
        "Year Established", min_value=1980, max_value=2024, value=2009, step=1
    )
    Store_Location_City_Type = st.selectbox("City Tier", options=store_loc_types)

# UI for capturing Product details
st.subheader("Product Details")
col3, col4 = st.columns(2)
with col3:
    Product_Id            = st.text_input("Product ID", value="FD6114")
    Product_Type          = st.selectbox("Product Type", options=product_types)
    Product_Sugar_Content = st.selectbox("Sugar Content", options=product_sugar_content_types)
with col4:
    Product_Weight         = st.number_input("Weight (kg)", min_value=0.1, max_value=50.0, value=12.66)
    Product_MRP            = st.number_input("Max Retail Price ($)", min_value=1.0, max_value=500.0, value=117.08)
    Product_Allocated_Area = st.number_input(
        "Allocated Display Area (ratio)", min_value=0.001, max_value=1.0, value=0.027, format="%.3f"
    )

if st.button("Forecast Sales", type="primary"):
    payload = {
        "Product_Id":               Product_Id,
        "Product_Weight":           float(Product_Weight),
        "Product_Sugar_Content":    Product_Sugar_Content,
        "Product_Allocated_Area":   float(Product_Allocated_Area),
        "Product_Type":             Product_Type,
        "Product_MRP":              float(Product_MRP),
        "Store_Id":                 Store_Id,
        "Store_Establishment_Year": int(Store_Establishment_Year),
        "Store_Size":               Store_Size,
        "Store_Location_City_Type": Store_Location_City_Type,
        "Store_Type":               Store_Type,
    }

    try:
        resp = requests.post(f"{API_URL}/predict", json=payload, timeout=10)

        # Safely attempt to parse JSON regardless of status code
        try:
            result = resp.json()
        except requests.exceptions.JSONDecodeError:
            st.error(
                f"API returned an unexpected response "
                f"(HTTP {resp.status_code}):\n\n{resp.text[:500]}"
            )
            st.stop()

        # check status AFTER we have the parsed body
        if resp.status_code != 200:
            st.error(f"API error ({resp.status_code}): {result.get('error', 'Unknown error')}")
            st.stop()

        st.success(
            f"Forecasted quarterly sales for **{result['Product_Id']}** "
            f"at store **{result['Store_Id']}**: "
            f"$ **{result['forecasted_sales']:,.2f}**"
        )

    except requests.exceptions.ConnectionError:
        st.error("Could not connect to the forecasting API. Is it running?")
    except requests.exceptions.Timeout:
        st.error("Request timed out — the API took too long to respond.")

Overwriting deployment_files/app.py


## Dependencies File

### Generate Requirements.txt from actual used Python packages

In [ ]:
%%writefile deployment_files/requirements.txt

flask>=2.3
streamlit>=1.35
pandas>=1.5
numpy>=1.24
scikit-learn>=1.3
xgboost>=2.0
joblib>=1.3
requests>=2.31
gunicorn>=21.2

Overwriting deployment_files/requirements.txt


## DockerFile

In [ ]:
%%writefile deployment_files/Dockerfile

# HuggingFace Spaces runs containers as a non-root user (uid=1000)
# so we must grant ownership of /app explicitly
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip3 install --no-cache-dir -r requirements.txt

# Give the non-root HF user permission to run the scripts
RUN chmod +x start.sh

# HuggingFace Spaces only exposes port 7860 — must use 7860 for Streamlit
EXPOSE 7860

CMD ["bash", "start.sh"]

Overwriting deployment_files/Dockerfile


In [ ]:
%%writefile deployment_files/start.sh

#!/bin/bash
set -e

# Start Flask API in the background on port 5000 (internal only)
python api.py &

# Wait for API to be ready before Streamlit starts
sleep 3

# HuggingFace Spaces requires port 7860 — this is the critical fix
streamlit run app.py \
    --server.port=7860 \
    --server.address=0.0.0.0 \
    --server.enableXsrfProtection=false \
    --server.enableCORS=false

Overwriting deployment_files/start.sh


In [ ]:
!ls "/content/drive/My Drive/Colab Notebooks/Model Deployment/SuperKart/deployment_files"

api.py	Dockerfile	  retail_chain_forecast_model_v1_0.joblib
app.py	requirements.txt  start.sh


## Uploading Files to Hugging Face Space (Streamlit Space)

In [ ]:
from google.colab import userdata
access_key = userdata.get("HF_TOKEN") ## Hugging Face token created from access keys in write mode
repo_id = "omsoni/retail_chain_sales_forecast"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/drive/My Drive/Colab Notebooks/Model Deployment/SuperKart/deployment_files",  # Local folder path in azureml
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...il_chain_forecast_model_v1_0.joblib: 100%|##########| 3.83MB / 3.83MB            

CommitInfo(commit_url='https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast/commit/76e3ffdb079ba6ceb170d872ed430d9629927e9b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='76e3ffdb079ba6ceb170d872ed430d9629927e9b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast', endpoint='https://huggingface.co', repo_type='space', repo_id='omsoni/retail_chain_sales_forecast'), pr_revision=None, pr_num=None)

### UI Integrated with Deployed model can be access at space endpoint below:

[Huggingface Space endpoint for Dockerized Forecasting Streamlit App and Final Model](https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast)

![](https://drive.google.com/uc?export=view&id=1J1Bvkazd3uHHfDjRmPEEhG_ST9EUCd_L)